### Part 4: The ”Budget Keeper” (Token Economics)

### 1. Configuration & Imports

In [ ]:
import sys
import os
import yaml

project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from utils.llm_client import LLMClient
from utils.prompts import render
from utils.token_utils import count_text_tokens

# Load from YAML to get provider and model
with open("../config/config.yaml", "r") as f:
    config_data = yaml.safe_load(f)
with open("../config/models.yaml", "r") as f:
    models_config = yaml.safe_load(f)

provider = config_data['providers']['default']
model = models_config[provider]['reason']

# Initialize Client
client = LLMClient(provider=provider, technique="reason")

### 2. Load Spam Data

In [ ]:
with open("../data/synthesized_for_part4.txt", "r", encoding="utf-8") as f: #incidents.txt also works fine
    # Assuming incidents are separated by newlines or double newlines
    incidents = [line.strip() for line in f if line.strip()]

### 3. The Budget Keeper Logic

In [ ]:
for i, raw_incident in enumerate(incidents[1:], 1):
    # Accurate token counting using the configured model
    token_count = count_text_tokens(raw_incident, provider=provider, model=model)
    
    # Check against the 150-token threshold
    if token_count > 150:
        print(f"🚨 [INCIDENT {i:02}] | Tokens: {token_count:<4} | STATUS: BLOCKED/TRUNCATED")
        
        # Summarize to preserve mission-critical info
        prompt_sum, _ = render("overflow_summarize.v1", message=raw_incident)
        res_summary = client.chat([{"role": "user", "content": prompt_sum}], temperature=0.0)
        
        print(f"✨ Mission-Critical Summary: {res_summary['text']}\n")
    else:
        print(f"✅ [INCIDENT {i:02}] | Tokens: {token_count:<4} | STATUS: PASSED\n")